In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Open by name (or use open_by_url / open_by_key if you have the sheet's URL/ID)
sheet = gc.open('Capstone Copy of Appointments').worksheet('Appointments_2025/2026')
tenure_sheet= gc.open('Capstone Copy of Appointments').worksheet('Clinician_Tenure')

# Pull all data into a DataFrame
import pandas as pd
import json
data = sheet.get_all_records()
raw_df = pd.DataFrame(data)

tenure = tenure_sheet.get_all_records()
tenure_df = pd.DataFrame(tenure)


In [ ]:
# Mount Drive first — everything below depends on this
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
Github_Token = userdata.get('Github_Token')
!git remote set-url origin https://{Github_Token}@github.com/Sharion2023/Data_Analytics_Capstone.git

# Move into the repo (already cloned, lives permanently in Drive)
%cd /content/drive/MyDrive/Data_Analytics_Capstone

# Git identity (session-only, still needs to be set each time)
!git config user.name "Sharion2023"
!git config user.email "your-email@example.com"

# Activate nbstripout locally each session
!pip install nbstripout --quiet
!nbstripout --install

# Load staff_anon.json from Drive
import json
with open('/content/drive/MyDrive/staff_anon.json', 'r') as f:
    staff_anon = json.load(f)

print("✅ Drive mounted, repo location set, git configured, nbstripout active, staff_anon loaded.")

In [ ]:
#create df for calculated results
si_ratio_sheet = gc.open('Capstone Copy of Appointments').worksheet('DataStudioSource_Practitioner')
si_data = si_ratio_sheet.get_all_records()
si_df = pd.DataFrame(si_data)

In [ ]:
# Strip whitespace on the join key in both DataFrames
raw_df['staff_member_name'] = raw_df['staff_member_name'].str.strip()
tenure_df['staff_member_name'] = tenure_df['staff_member_name'].str.strip()
si_df['staff_member_name'] = si_df['staff_member_name'].str.strip()

# Merge
df = raw_df.merge(tenure_df, on='staff_member_name', how='left')

print(f"raw_df rows: {len(raw_df)} | merged df rows: {len(df)}")

In [ ]:
unmatched_count = df[df['start_date'].isna()]['staff_member_name'].nunique()
print(f"{unmatched_count} unique staff members did not match")

In [ ]:
df = df.merge(si_df, on=['staff_member_name', 'iso_week'], how='left')

In [ ]:
#map to aonymous mapping
df['clinician_id'] = df['staff_member_name'].map(staff_anon)

In [ ]:
#check that all mapping worked
unmatched = df['clinician_id'].isna().sum()
print(f"{unmatched} rows failed to map to a clinician_id")

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:

df

In [ ]:
df_clean = df.copy()

In [ ]:
df_clean.columns

In [ ]:
#drop unnecessary columns
col_to_keep = [
    'patient_number',    # this is your patient ID field, not 'patient_id'
    'clinician_id',
    'start_at',
    'arrived_at',
    'first_visit',
    'treatment_name',
    'booked_at', # lead time analysis for intake-conversion stream
    'start_date',
    'iso_week',
    'subsequent_visits',
    'initial_visits',
    'real_date',
       'weekly_S/I_ratio',
    'rolling_4-week_S/I_ratio',
    'tenure_status',
       'unique_patients',
    'fall_off_patients',
    '4_wk_fall_off_patient_calc'
]

df_clean = df_clean[col_to_keep]

In [ ]:
df_clean.head()
df_clean.isna().sum()


In [ ]:
df_clean['fall_off_patients'].unique()

In [ ]:
df_clean['start_date'] = pd.to_datetime(df_clean['start_date'], errors='coerce')
df_clean['start_at'] = pd.to_datetime(df_clean['start_at'], errors='coerce')
df_clean['arrived_at'] = pd.to_datetime(df_clean['arrived_at'],errors='coerce')
df_clean['booked_at'] = pd.to_datetime(df_clean['booked_at'], errors='coerce')
df_clean['real_date'] = pd.to_datetime(df_clean['real_date'], errors='coerce')
df_clean['weekly_S/I_ratio'] = pd.to_numeric(df_clean['weekly_S/I_ratio'], errors='coerce')
df_clean['rolling_4-week_S/I_ratio'] = pd.to_numeric(df_clean['rolling_4-week_S/I_ratio'], errors='coerce')
df_clean['fall_off_patients'] = pd.to_numeric(df_clean['fall_off_patients'], errors='coerce')
df_clean['4_wk_fall_off_patient_calc'] = pd.to_numeric(df_clean['4_wk_fall_off_patient_calc'], errors='coerce')

In [ ]:
df_clean.dtypes

In [ ]:
df_clean.sort_values('real_date', ascending=True).tail()

In [ ]:
df_clean.head(10)

In [ ]:
df_clean.shape

In [ ]:
df_clean['clinician_id'].unique()

In [ ]:
# Confirm unique patient count, expect 4309
print(df_clean['patient_number'].nunique())

In [ ]:
clinic_wide_sheet = gc.open('Capstone Copy of Appointments').worksheet('DataStudioSource_Clinic')
clinic_wide_data = clinic_wide_sheet.get_all_records()
clinic_wide_df = pd.DataFrame(clinic_wide_data)

In [ ]:
import matplotlib.pyplot as plt

#Calculate clinic wide S/I ratio

clinic_wide_df['rolling_4-week_S/I_ratio'] = pd.to_numeric(
    clinic_wide_df['rolling_4-week_S/I_ratio'], errors='coerce'
)

clinic_trend = clinic_wide_df.set_index('iso_week')['rolling_4-week_S/I_ratio']

fig, ax = plt.subplots(figsize=(10, 5))

clinic_trend.plot(ax=ax, marker='o', color='#4C72B0')
plt.title('Clinic 4 Week S/I Ratio Trend')
plt.xlabel('Week')
plt.ylabel('4 Week S/I Ratio')
plt.xticks(rotation=45)
plt.axhline(
    y=4, color='orange', linestyle='--', linewidth=2, label='Goal: 4'
)
plt.legend()
plt.show()

In [ ]:
df_clean['rolling_4-week_S/I_ratio'] = pd.to_numeric(
    df_clean['rolling_4-week_S/I_ratio'], errors='coerce'
)

practitioner_trend = df_clean.groupby('clinician_id')['rolling_4-week_S/I_ratio'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))

practitioner_trend.plot(kind ='barh', ax=ax, color='#4C72B0')
plt.title('Practitioner Average 4 Week S/I Ratio Trend')
plt.xlabel('S/I Ratio')
plt.ylabel('Practitioner')
plt.xticks(rotation=45)
plt.axvline(
    x=4, color='orange', linestyle='--', linewidth=2, label='Goal: 4'
)
plt.legend()
plt.show()

In [ ]:
df_clean['rolling_4-week_S/I_ratio'].dtype

In [ ]:
# Build chronological date to use repeatedly, utilize real date to sort and index iso_week.
week_order = df_clean.groupby('iso_week')['real_date'].min().sort_values().index

fig, ax = plt.subplots(figsize=(12, 6))

for clinician in practitioner_trend.index:
    #build our group
    group = df_clean[df_clean['clinician_id'] == clinician]
    #calculate weekly rolling average, should start after 4 weeks of data
    weekly = group.groupby('iso_week')['rolling_4-week_S/I_ratio'].mean()
    #bring in chronological date variable to ensure correct order
    weekly = weekly.reindex(week_order)
    weekly.plot(ax=ax, marker='o', label=clinician, alpha=0.7)

ax.set_xlabel('Week')
ax.set_ylabel('Rolling 4-Week S/I Ratio')
ax.set_title('S/I Ratio Trend by Practitioner Over Time')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.xticks(rotation=45)
plt.axhline(
    y=4, color='black', linestyle='--', linewidth=2, label='Goal: 4'
)
plt.tight_layout()
plt.show()


This graphic appears chaotic and noisy, which is precisely why giving the business owner the ability to sort by practioners (as requested) was a top priority in the visualizations.

In [ ]:
df_clean['start_date'].dtype

In [ ]:
!git status

In [ ]:
!git add 'SI_Ratio_Analysis.ipynb'
!git commit -m 'Testing graphing possiblilities for practitioner S/I ratio, x-axis label shifted.'
!git push

In [ ]:
visits_per_clinician = df_clean.groupby('clinician_id').size()
print(visits_per_clinician)
print(visits_per_clinician.min())
print(visits_per_clinician.max())

In [ ]:
#Calculate each practitioner's tenure with the clinic

reference_date = df_clean['real_date'].max()
df_clean['tenure_years'] = (reference_date - df_clean['start_date']).dt.days / 365.25

#Build summary table of all clinicians
tenure_si_summary = df_clean.groupby('clinician_id').agg(
    tenure_years=('tenure_years', 'first'),
    avg_si_ratio=('rolling_4-week_S/I_ratio', 'mean'),
    patients_per_clinician =('patient_number', 'nunique')
).reset_index()

print(tenure_si_summary)

In [ ]:
#Looking at clinician I, I want to see if low patient count correlates to extreme S/I ratio
print(tenure_si_summary[['patients_per_clinician', 'avg_si_ratio']].sort_values('patients_per_clinician'))

In [ ]:
tenure_si_summary['avg_si_ratio'].describe()

There doesn't necessarily appear to be a correlation. However, two clinicians do appear to be outliers, throwing the averages off.

In [ ]:
# Z-score method: flag clinicians whose avg_si_ratio is unusually far from the group mean
mean_ratio = tenure_si_summary['avg_si_ratio'].mean()
std_ratio = tenure_si_summary['avg_si_ratio'].std()

tenure_si_summary['z_score'] = (tenure_si_summary['avg_si_ratio'] - mean_ratio) / std_ratio

print(tenure_si_summary[['clinician_id', 'patients_per_clinician', 'avg_si_ratio', 'z_score']].sort_values('z_score', ascending=False))

Z-Score in statistics measures how many standard deviations a data point lies away from the mean of a distribution. It standardizes values across different distributions, enabling meaningful comparisons even when datasets have different means and standard deviations. It is widely used in hypothesis testing, outlier detection and normalizing data for machine learning models.

https://www.geeksforgeeks.org/data-science/z-score-in-statistics/

In [ ]:
Z_THRESHOLD= 2


In [ ]:
tenure_si_summary['patients_per_clinician'].describe()

In [ ]:
#Looking at the description, there are three distinct tiers of practitioners related to visit

In [ ]:
Q1 = summary['patient_count'].quantile(0.25)
Q3 = summary['patient_count'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = summary[(summary['patient_count'] < lower_bound) | (summary['patient_count'] > upper_bound)]
print(f"Bounds: {lower_bound:.0f} to {upper_bound:.0f}")
print(outliers)

IQR method not effective on such a small data set. Will attempt

In [ ]:
MIN_CASELOAD = 50  # justify this number in your methodology write-up

reliable_summary = summary[summary['patient_count'] >= MIN_CASELOAD]
excluded = summary[summary['patient_count'] < MIN_CASELOAD]

print(f"Included: {len(reliable_summary)} clinicians")
print(f"Excluded (caseload < {MIN_CASELOAD}): {len(excluded)} clinicians")
print(excluded)

In [ ]:
from scipy import stats

# --- Full dataset (all 16 clinicians) ---
corr_full, p_full = stats.pearsonr(summary['patient_count'], summary['retention_rate'])
print("=== Full dataset (all clinicians) ===")
print(f"n = {len(summary)}")
print(f"Pearson r = {corr_full:.3f}, p = {p_full:.3f}")

print()

# --- Thresholded dataset (caseload >= MIN_CASELOAD) ---
corr_thresh, p_thresh = stats.pearsonr(reliable_summary['patient_count'], reliable_summary['retention_rate'])
print(f"=== Thresholded dataset (caseload >= {MIN_CASELOAD}) ===")
print(f"n = {len(reliable_summary)}")
print(f"Pearson r = {corr_thresh:.3f}, p = {p_thresh:.3f}")